# Chapter 16 — Supplement 4: `flag_regulatory`

*The model proposes, a trained graph verifies and corrects, a fine-tuned embedding reads the message geometrically, and the graph decides escalation.*

Companion to `16_capstone.ipynb`. This is the chapter's clearest example of **GMS as a governance backstop**. Qwen proposes which regulations a complaint implicates; a separately trained and *calibrated* GMS store checks each proposal against the evidence actually in the message, corrects mislabels, **reads the message geometrically through a fine-tuned embedding** to catch the high-severity flags a keyword would miss, and decides escalation by walking the graph.

## Where it fits

Step 4 of the workflow, and the gate to human escalation: when it returns `escalate=True`, the agent escalates instead of drafting.

| | |
| --- | --- |
| **Backed by** | Qwen proposer + a trained, calibrated GMS regulatory store + a fine-tuned-embedding cap reader |
| **Artifact** | `data/gms_regulatory_store/` + `calibration.json` (theta); `data/gms_regulatory_cap/` (the fine-tuned embedding + per-flag caps) |
| **Build scripts** | `scripts/build_regulatory_guard_store.py`, `scripts/calibrate_regulatory_guard.py`, `scripts/build_regulatory_cap_store.py` |
| **Modules** | `agentlab/models/qwen_flagger.py` + `agentlab/capstone/regulatory_guard.py` + `agentlab/capstone/manifold_evidence.py` |
| **Risk level** | `LOW` |

The store-and-graph backstop **fails loud**: if it or its calibration is missing, the tool raises (and the agent escalates to a human) rather than silently downgrading. The geometric reader, by contrast, fails **soft**: if its cap artifact or encoder is unavailable it simply returns nothing and the guard falls back to the lexicon-and-graph path.

In [ ]:
import os, json, warnings, contextlib, io
from pathlib import Path

os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
os.environ.setdefault('KNOWLYTIX_EULA_ACCEPTED', '1')  # silence the EULA reminder line
warnings.filterwarnings('ignore')  # quiet the harmless GPU-capability / HF notices

# knowlytix prints a one-line license banner (it names the licensee) to stderr
# on first import. The package has no flag to disable it, so import it once here
# with stderr/stdout captured; every later import is a cache hit and stays quiet,
# so the banner never lands in a baked cell output.
with contextlib.redirect_stderr(io.StringIO()), contextlib.redirect_stdout(io.StringIO()):
    try:
        import knowlytix.core  # noqa: F401
    except Exception:
        pass

# The data dir lives under code/ after the trilogy reorg; find the dir that
# holds it whether the notebook is launched from notebooks/, the repo root,
# or code/ itself.
root = next((c for c in (Path('.'), Path('..'), Path('../code'), Path('code'))
             if (c / 'data').exists()), None)
assert root is not None, 'could not locate the code/ data dir (run from notebooks/ or repo root)'
print('repo root:', root.resolve())

## 1. The proposer — Qwen suggests flags

`get_default_flagger().propose(message, product, issue)` returns a list drawn from `{UDAAP, Reg_X, Reg_E, Reg_Z, FCRA}`, or `None` on failure. This is *recall-oriented* and untrusted — it is the input the graph will verify.

In [ ]:
from agentlab.models.qwen_flagger import get_default_flagger, ALLOWED_FLAGS

print('allowed flags:', ALLOWED_FLAGS)
flagger = get_default_flagger()

msg = 'You charged me an unauthorized $50 overdraft fee and refuse to fix it.'
proposed = flagger.propose(msg, product='checking_account', issue='overdraft_fee')
print('\nmessage :', msg)
print('proposed:', proposed)

The model proposes one or more flags. We do **not** act on this directly — a generative proposer can over- or under-flag. The graph decides what stands.

## 2. The backstop — a calibrated GMS store verifies and corrects

`get_default_guard()` (in `regulatory_guard.py`) loads the trained store and the calibrated threshold theta from `calibration.json`. `verify_and_correct(proposed, message)` does two things:

- **Verify** — keep a Qwen proposal only if the graph supports it from the message's evidence.
- **Correct** — derive *every* flag the evidence supports from the graph, regardless of what Qwen said.

A flag-evidence link counts as supported iff its geometric score is at or below theta.

In [ ]:
# Name clash note: regulatory_guard and semantic_guard both export get_default_guard;
# we alias this one as the *regulatory* guard.
from agentlab.capstone.regulatory_guard import get_default_guard as get_reg_guard

guard = get_reg_guard()
print('calibrated theta:', guard.theta)

verdict = guard.verify_and_correct(proposed or [], msg)
for k, v in verdict.items():
    print(f'  {k:14s}: {v}')

**Reading the verdict.** `evidence` is what the alias lexicon found in the message; `graph_derived` are flags the graph itself supports from that evidence (the *correction* path); `manifold_flags` are the high-severity flags the fine-tuned embedding read straight from the message (Section 2b); `verified` are Qwen proposals the graph confirmed; `rejected` are proposals it could not support; `flags` is the sanctioned union; and `escalate` plus `severity_paths` come from the traversal in the next cell. The graph — not the model, and not a constant in code — has the final say.

## 2b. Reading the message geometrically

The alias lexicon can only match phrases it knows: a customer who calls a charge *'a trick'* or *'not right'* carries a real UDAAP grievance with no canonical token. So for the two high-severity flags the guard reads the message **geometrically**. A small supervised fine-tune — a rank-1 *rotation* adapter on a frozen MiniLM encoder, trained with a prototype objective and a drift penalty that keeps it near the identity — reshapes the embedding so messages cluster by regulation. The tuned vectors are inserted into GMS via `EmbeddingConfig` (with `out_dim` set to the store's `d_v`, so the reduction is the manifold's own Stiefel projection rather than a lossy truncation), and a spherical cap is trained around each flag's `has_evidence` cluster. A flag fires when the message falls within a calibrated geodesic radius of its cap center. `scripts/build_regulatory_cap_store.py` builds the artifact; `ManifoldFlagScorer` loads and applies it.

In [ ]:
from agentlab.capstone.manifold_evidence import get_default_scorer

scorer = get_default_scorer()   # None if the cap artifact is unavailable (degrade-safe)
if scorer is None:
    print('(gms_regulatory_cap/ not found — run scripts/build_regulatory_cap_store.py; '
          'or run 00_setup.ipynb Tier 2 to build all artifacts)')
else:
    print('per-flag radius (tau):', {k: round(v, 3) for k, v in scorer.thresholds.items()})

    for m in ['I am outraged. My overdraft fee is unfair and I demand a refund.',
              'Just waive my overdraft fee, I do not want to hear any policies.']:
        dd = {k: round(x, 2) for k, x in scorer.flag_distances(m).items()}
        print(f'  fired={sorted(scorer.flags_for_message(m))}  dist={dd}')
        print(f'     <- {m!r}')


The unfair-fee complaint falls inside the UDAAP cap and fires; the bare *'just waive my fee'* demand — adversarial, but not an allegation that the fee is unfair — stays outside it and does not. That is the regulatory distinction, recovered from the message's position in the fine-tuned space rather than from any keyword the lexicon happens to hold.

## 3. Escalation as a multi-hop traversal

Whether a flagged case escalates is itself a graph question. `escalation_for_flags` walks two hops from each fired flag — `flag --has_severity--> severity --has_action--> action` — and escalates iff some path lands on the `escalate` action. This is deterministic graph traversal, identical every run.

In [ ]:
from agentlab.capstone.regulatory_guard import _FLAG_TO_ENTITY
print('flag -> entity map:', _FLAG_TO_ENTITY)

for flags in (['UDAAP'], ['Reg_E'], ['Reg_X'], ['Reg_Z'], ['FCRA']):
    escalate, paths = guard.escalation_for_flags(flags)
    path = paths[0] if paths else {}
    chain = (f"{path.get('flag')} -> {path.get('severity')} -> {path.get('action')}"
             if path else '(no path)')
    print(f'  {flags[0]:6s} escalate={escalate!s:5s}  [{chain}]')

`UDAAP` and `Reg_X` carry `high` severity and route to `escalate`; `Reg_E`, `Reg_Z` and `FCRA` are `standard` and route to `draft`. So a transaction dispute (Reg_E) still gets a drafted provisional-credit response, while a UDAAP fee demand escalates — and *changing that policy is an edit to the compendium followed by a retrain, never a code change.*

## 4. The governed tool end to end

`flag_regulatory.fn` combines a deterministic rule baseline, the Qwen proposal, and the GMS verify/correct, returning the sanctioned flags, the escalation decision, and the traversed severity paths (which the audit log records). We run it on a real eval case.

In [ ]:
from agentlab.capstone.banking_tools import flag_regulatory

cases = json.loads((root / 'data' / 'eval_cases' / 'cases.json').read_text())
case = next(c for c in cases if c['id'] == 'case-016')   # 'my overdraft fee is unfair'
args = flag_regulatory.input_schema(
    product='checking_account', issue='overdraft_fee', message=case['message'])
out = flag_regulatory.fn(**args.model_dump())
print('message       :', case['message'])
print('flags         :', out['flags'])
print('escalate      :', out['escalate'])
for p in out['severity_paths']:
    print('  path        :', f"{p['flag']} -> {p['severity']} -> {p['action']}")

case-016 alleges the overdraft fee is *unfair* and demands a refund; the geometric reader places it inside the UDAAP cap, the flag fires, and the severity path traverses to `escalate` — so the harness escalates to a human rather than drafting a unilateral remedy, exactly as the capstone notebook shows end to end. A bare *'just waive my fee'* demand (case-008), by contrast, is adversarial but not a UDAAP allegation, and does not escalate.

## 5. How the store was built and calibrated

Two scripts produced `data/gms_regulatory_store/`:

**Build** — `scripts/build_regulatory_guard_store.py` ingests `data/regulatory_guard.md` (flag-evidence links, evidence aliases, flag severity/action) and trains the GMS store (`config.train.epochs = 600`, `lr = 5e-3`).

**Calibrate** — `scripts/calibrate_regulatory_guard.py` builds a labeled cohort from the store itself (every real `has_evidence` triple is a positive; each evidence entity paired with a *wrong* flag is a negative), sweeps the threshold theta, and picks the value that maximizes accuracy subject to a 5% false-allow ceiling — writing it to `calibration.json`.

```bash
python scripts/build_regulatory_guard_store.py
python scripts/calibrate_regulatory_guard.py
```

The shipped calibration:

In [ ]:
calib = json.loads((root / 'data' / 'gms_regulatory_store' / 'calibration.json').read_text())
print(json.dumps(calib, indent=2))

The calibrated `evidence_threshold` (~1.22) is the operating point at which the cohort reaches 100% accuracy with a 0% false-allow rate. It is data, not code: re-running calibration after a store change re-derives it.

## Summary

- `flag_regulatory` = **Qwen proposes, a trained+calibrated GMS graph disposes**. The graph verifies proposals against in-message evidence and *corrects* by deriving every supported flag itself.
- Escalation is a two-hop graph traversal (`flag -> severity -> action`), so the escalation policy lives in the calibrated compendium, not in code.
- The backstop is required and fails loud; theta comes from `scripts/calibrate_regulatory_guard.py`.

Next: **Supplement 5 — `draft_response`**, a LoRA writer with a GMS draft verifier.